## Optimised code for inference

In [1]:
# Cell 1: Setup and Model Loading
%%capture
!pip install transformers gradio bitsandbytes peft accelerate


In [ ]:
import torch
import gradio as gr
from threading import Thread
from transformers import BitsAndBytesConfig, AutoTokenizer, AutoModelForCausalLM, TextIteratorStreamer
from peft import PeftModel

BASE_MODEL_ID = "large-traversaal/Alif-1.0-8B-Instruct"
ADAPTER_ID = "hamza-amin/alif-emergency-finetuned"

# 4-bit quantization configuration
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto"
)
model = PeftModel.from_pretrained(base_model, ADAPTER_ID)

In [ ]:
# model.eval()

In [5]:
# Cell 2: Inference and Gradio Interface
chat_prompt = """You are a Rescue 1122 emergency operator in Pakistan. Respond in professional Urdu with ONE brief response only.

### User:
{prompt}

### Assistant:"""

def generate_response(query):
    prompt = chat_prompt.format(prompt=query)
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda" if torch.cuda.is_available() else "cpu")

    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

    generation_kwargs = dict(
        inputs,
        streamer=streamer,
        max_new_tokens=100,  # Reduced from 256
        do_sample=True,
        top_p=0.9,
        top_k=50,
        temperature=0.7,
        repetition_penalty=1.15,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
        # Stop at these sequences
        stopping_criteria=None,
    )

    thread = Thread(target=model.generate, kwargs=generation_kwargs)
    thread.start()

    generated_text = ""
    for new_text in streamer:
        # Stop if model starts a new user turn
        if "### User:" in new_text or "###" in new_text:
            break
        if new_text.endswith(tokenizer.eos_token):
            new_text = new_text[:len(new_text) - len(tokenizer.eos_token)]
        generated_text += new_text
        yield generated_text

    # Clean up any partial ### markers
    generated_text = generated_text.split("###")[0].strip()
    yield generated_text

iface = gr.Interface(
    fn=generate_response,
    inputs=gr.Textbox(lines=2, placeholder="اپنا ایمرجنسی سوال یہاں لکھیں..."),
    examples=[
        'میں کراچی سے بول رہا ہوں۔ یہاں آگ لگی ہے۔',
        'میری بہن کو دل کا دورہ پڑا ہے۔ مدد چاہیے۔',
        'سیلاب کا پانی گھر میں آ گیا ہے۔',
        'سڑک پر حادثہ ہوا ہے۔ لوگ زخمی ہیں۔',
        'گیس کا لیکج ہو رہا ہے۔ کیا کریں؟',
        'بجلی کی تار گر گئی ہے۔'
    ],
    outputs="text",
    title="🚨 Rescue 1122 Emergency Response System",
    description="Urdu Emergency Assistant - Powered by Fine-tuned Alif Model",
)

iface.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5a53c881230eea3c2d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
